# 3a. Contract negotiation, step by step (DSP 2025-1)

A contract negotiation turns an offer the Provider published into an agreement both sides hold.
This notebook walks the full exchange message by message, with one counter-offer round, so every
state of the negotiation appears:

| Step | Sender | RPC | DSP message (endpoint section) | State |
|---|---|---|---|---|
| 2 | Consumer | `setup-request-init` | `ContractRequestMessage` (DSP 8.2.2) | `REQUESTED` |
| 3 | Provider | `setup-offer` | `ContractOfferMessage` (DSP 8.3.4) | `OFFERED` |
| 4 | Consumer | `setup-request` | `ContractRequestMessage` (DSP 8.2.3) | `REQUESTED` |
| 5 | Provider | `setup-offer` | `ContractOfferMessage` (DSP 8.3.4) | `OFFERED` |
| 6 | Consumer | `setup-acceptance` | `ContractNegotiationEventMessage` `ACCEPTED` (DSP 8.2.4) | `ACCEPTED` |
| 7 | Provider | `setup-agreement` | `ContractAgreementMessage` (DSP 8.3.5) | `AGREED` |
| 8 | Consumer | `setup-verification` | `ContractAgreementVerificationMessage` (DSP 8.2.5) | `VERIFIED` |
| 9 | Provider | `setup-finalization` | `ContractNegotiationEventMessage` `FINALIZED` (DSP 8.3.6) | `FINALIZED` |

**Runs on its own.** It leaves `agreement_id` for the transfer notebooks, and records
`agreement_source = ("rpc", target_id)` so that a transfer notebook can tell whether the agreement
in the kernel belongs to the loaded dataset.

## 1. Prerequisites

The guard populates a catalog only when no catalog notebook has run in this kernel; it runs
`setup_catalog_policies.ipynb`, which in turn onboards the participants if needed. Run
`setup_catalog_policies_oauth.ipynb` first to negotiate on its dataset instead.

In [2]:
if not globals().get("catalog_setup"):
    %run ./setup_catalog_policies.ipynb

Authority URL: http://127.0.0.1:1500
Consumer URL:  http://127.0.0.1:1100
Provider URL:  http://127.0.0.1:1200
Wallet linked for Authority (http://127.0.0.1:1500)
Wallet linked for Consumer (http://127.0.0.1:1100)
Wallet linked for Provider (http://127.0.0.1:1200)
Authority DID: did:web:127.0.0.1%3A1500

Consumer DID:  did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ5cS12NXdJNFM2MTl2SnJITmpENDVra0owUGw4VGlBTzZYTnFJc1o4SDBaTEs5T2JoWTBTeGtEQ2JlZWd2RlZEU0MzM182N3lVZVpUWFI2SjFXUExHdHNtLU51SXJtR1RVRVgteFA0WnFLVGRFUTFSdUV0MExtMUhtdkdJWG1zLVJrTXRNVjcxRWt0MjNiZFYzX01WTjRUOUd5WnlaQTB6RDRlTFUxRTRTRzlRMGl4MGFRbC16eGdsVFRYQ2l4VTdCTzRXcHRkaUp3a0Jid2hMbnVqeXFzSmNvQXpPU000aXZtZlRTVW1Yd0Nrc1gySTFxVTNnVWdtUGFtbmEta3ZUb3FYZUk4ODNlRTFkSWNuUG53dmJlRmpfa0xuQ2JrNzhjNkVkRXZ0OVBMa3ByTWZfdmVxeVlOWjB0WlpYUGM5c0lRVlMyUG5xZkN5RUhsWWVuMXNXclEifQ

Provider DID:  did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dE

### How the negotiation RPCs work

Neither participant calls the peer's DSP endpoints directly. Each one asks **its own agent** to
send the next DSP message through an RPC under `/dsp/current/negotiations/rpc/`. The body is
eunomia JSON-LD:

```json
{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcNegotiation<Kind>Message",
  "consumerPid": "...", "providerPid": "...",
  "offer": { ... }
}
```

The agent reads it through RDF, builds the DSP message, checks it against the negotiation state
machine and its own role, sends it with its peer token and stores the result once the peer
accepts it. Every RPC answers an `RpcNegotiationResult`:
- `request`: the RPC body as received.
- `response`: the peer's answer (its `ContractNegotiation` ACK for an opening message).
- `negotiationAgentModel`: the caller's own process as it stands, with `state`, `identifiers`,
  `messages`, `offers` and, once agreed, `agreement`.

An opening RPC answers 201, the rest 200. Errors answer a `ContractNegotiationError`: 400 for a
broken rule, 404 for an unknown negotiation, 409 for a stale or repeated message, 502 when the
peer refused it. A continuation names the negotiation by `providerPid` or `consumerPid`; these
notebooks send both.

In [3]:
def negotiation_step(response, expected_state):
    """Prints an RpcNegotiationResult and checks the caller's process reached expected_state."""
    answer = show(response)
    process = answer["negotiationAgentModel"]
    assert process["state"] == expected_state, f"expected {expected_state}, got {process['state']}"
    print(f"\n==> {process['state']}")
    return process

## 2. Contract request (Consumer -> Provider)

The Consumer opens the negotiation on the published offer: `@id` is the offer from the catalog
and `target` the dataset that publishes it. The Provider rejects a request on an offer it did not
publish, or on another dataset. `providerAddress` is where the request goes and `callbackAddress`
where the Provider answers.

In [5]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationRequestInitMessage",
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "offer": {
        "@id": policy_id,
        "@type": "Offer",
        "target": target_id,
        "permission": [{"action": "use"}],
    },
}
response = requests.post(f"{data_space_consumer}/dsp/current/negotiations/rpc/setup-request-init", headers=json_header, json=payload)
consumer_process = negotiation_step(response, "REQUESTED")
cn_consumer_id = consumer_process["identifiers"]["consumerPid"]
cn_provider_id = consumer_process["identifiers"]["providerPid"]
print(f"consumerPid: {cn_consumer_id}\nproviderPid: {cn_provider_id}")

HTTP 201 POST http://127.0.0.1:1100/dsp/current/negotiations/rpc/setup-request-init
{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcNegotiationResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcNegotiationRequestInitMessage",
    "associatedAgentPeer": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dENkJmbWk4OVhvV21ZTUhqSXI2Q1VPdUk2M2JSY09FakVmMnpvWEJockMtR2ViRS1qSk5zNTFQWTZ0MUYwWFIzMTVLczlqLWdHSzZ1MGxTSXMySkdXWFBxVFV2QnlTVzhpeE5MNzBjaHM4OXItZnRvZDRDakg2Mnlod2IzYk1WeWxrTjNIYnZWQVlkRXRpM1FnUXlwRFRNMTViSlVvUy1hZlZRUGRIMlYxbXA3X2FJY2FEUEhTVG1xMUhZQTgxc01lZm9mNTdfRFJWQUxrcXBSTzVEa2hSOTFfeEZqQXRCODEtSnRySHN2c19HZ0lvN24yUnh6MGZScnV4VkNQWlBtVVRDMncifQ",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "offer": {
      "@id": "urn:odrl-policy:bc1df9bf-1396-4d81-a5b9-33992b49db07",
      "@type

## 3. Counter-offer (Provider -> Consumer)

The Provider answers with different terms: use is limited to 1000 requests.

In [6]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationOfferMessage",
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,
        "@type": "Offer",
        "target": target_id,
        "permission": [{
            "action": "use",
            "constraint": [{"leftOperand": "count", "operator": "lteq", "rightOperand": "1000"}],
        }],
    },
}
response = requests.post(f"{data_space_provider}/dsp/current/negotiations/rpc/setup-offer", headers=json_header, json=payload)
provider_process = negotiation_step(response, "OFFERED")

HTTP 200 POST http://127.0.0.1:1200/dsp/current/negotiations/rpc/setup-offer
{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcNegotiationResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcNegotiationOfferMessage",
    "consumerPid": "urn:uuid:b796fe3f-2055-40f6-a172-fd3d91164704",
    "offer": {
      "@id": "urn:odrl-policy:bc1df9bf-1396-4d81-a5b9-33992b49db07",
      "@type": "Offer",
      "permission": [
        {
          "action": "use",
          "constraint": [
            {
              "leftOperand": "count",
              "operator": "lteq",
              "rightOperand": "1000"
            }
          ]
        }
      ],
      "target": "urn:dataset:17579bc9-2d22-410e-b15b-4de7368d8b63"
    },
    "providerPid": "urn:uuid:128098af-89fb-43f8-932e-4fe27cae69b9"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",


## 4. Counter-request (Consumer -> Provider)

The Consumer does not accept the limit and asks again for unrestricted use.

In [7]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationRequestMessage",
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,
        "@type": "Offer",
        "target": target_id,
        "permission": [{"action": "use"}],
    },
}
response = requests.post(f"{data_space_consumer}/dsp/current/negotiations/rpc/setup-request", headers=json_header, json=payload)
consumer_process = negotiation_step(response, "REQUESTED")

HTTP 200 POST http://127.0.0.1:1100/dsp/current/negotiations/rpc/setup-request
{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcNegotiationResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcNegotiationRequestMessage",
    "consumerPid": "urn:uuid:b796fe3f-2055-40f6-a172-fd3d91164704",
    "offer": {
      "@id": "urn:odrl-policy:bc1df9bf-1396-4d81-a5b9-33992b49db07",
      "@type": "Offer",
      "permission": [
        {
          "action": "use"
        }
      ],
      "target": "urn:dataset:17579bc9-2d22-410e-b15b-4de7368d8b63"
    },
    "providerPid": "urn:uuid:128098af-89fb-43f8-932e-4fe27cae69b9"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:uuid:b796fe3f-2055-40f6-a172-fd3d91164704",
    "providerPid": "urn:uuid:128098af-89fb-43f8-932e-4fe27cae69b9",
    "state": "REQUESTED"
  },
  "neg

## 5. Final offer (Provider -> Consumer)

The Provider gives in and offers the terms the Consumer asked for. This is the offer the agreement
will carry.

In [8]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationOfferMessage",
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,
        "@type": "Offer",
        "target": target_id,
        "permission": [{"action": "use"}],
    },
}
response = requests.post(f"{data_space_provider}/dsp/current/negotiations/rpc/setup-offer", headers=json_header, json=payload)
provider_process = negotiation_step(response, "OFFERED")

HTTP 200 POST http://127.0.0.1:1200/dsp/current/negotiations/rpc/setup-offer
{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcNegotiationResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcNegotiationOfferMessage",
    "consumerPid": "urn:uuid:b796fe3f-2055-40f6-a172-fd3d91164704",
    "offer": {
      "@id": "urn:odrl-policy:bc1df9bf-1396-4d81-a5b9-33992b49db07",
      "@type": "Offer",
      "permission": [
        {
          "action": "use"
        }
      ],
      "target": "urn:dataset:17579bc9-2d22-410e-b15b-4de7368d8b63"
    },
    "providerPid": "urn:uuid:128098af-89fb-43f8-932e-4fe27cae69b9"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:uuid:b796fe3f-2055-40f6-a172-fd3d91164704",
    "providerPid": "urn:uuid:128098af-89fb-43f8-932e-4fe27cae69b9",
    "state": "OFFERED"
  },
  "negotiati

## 6. Acceptance (Consumer -> Provider)

The Consumer accepts the last offer. The body only names the negotiation: `@type` says which
message goes out.

In [9]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationAcceptanceMessage",
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
}
response = requests.post(f"{data_space_consumer}/dsp/current/negotiations/rpc/setup-acceptance", headers=json_header, json=payload)
consumer_process = negotiation_step(response, "ACCEPTED")

HTTP 200 POST http://127.0.0.1:1100/dsp/current/negotiations/rpc/setup-acceptance
{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcNegotiationResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcNegotiationAcceptanceMessage",
    "consumerPid": "urn:uuid:b796fe3f-2055-40f6-a172-fd3d91164704",
    "providerPid": "urn:uuid:128098af-89fb-43f8-932e-4fe27cae69b9"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:uuid:b796fe3f-2055-40f6-a172-fd3d91164704",
    "providerPid": "urn:uuid:128098af-89fb-43f8-932e-4fe27cae69b9",
    "state": "ACCEPTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:65cec5b0-0cf3-4e89-9b64-7f90b2fc29b1",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Private",
    "state": "ACCEPTED",
    "stateAttribute": null,
    "associatedAgentPeer"

## 7. Agreement (Provider -> Consumer)

The Provider turns the accepted offer into a `ContractAgreementMessage` and sends it.

In [10]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationAgreementMessage",
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
}
response = requests.post(f"{data_space_provider}/dsp/current/negotiations/rpc/setup-agreement", headers=json_header, json=payload)
provider_process = negotiation_step(response, "AGREED")

HTTP 200 POST http://127.0.0.1:1200/dsp/current/negotiations/rpc/setup-agreement
{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcNegotiationResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcNegotiationAgreementMessage",
    "consumerPid": "urn:uuid:b796fe3f-2055-40f6-a172-fd3d91164704",
    "providerPid": "urn:uuid:128098af-89fb-43f8-932e-4fe27cae69b9"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:uuid:b796fe3f-2055-40f6-a172-fd3d91164704",
    "providerPid": "urn:uuid:128098af-89fb-43f8-932e-4fe27cae69b9",
    "state": "AGREED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:72a5343b-0126-4918-ac86-82d3150abe9a",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Public",
    "state": "AGREED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:

## 8. Verification (Consumer -> Provider)

The Consumer confirms it received the agreement and accepts it as binding.

In [11]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationVerificationMessage",
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
}
response = requests.post(f"{data_space_consumer}/dsp/current/negotiations/rpc/setup-verification", headers=json_header, json=payload)
consumer_process = negotiation_step(response, "VERIFIED")

HTTP 200 POST http://127.0.0.1:1100/dsp/current/negotiations/rpc/setup-verification
{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcNegotiationResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcNegotiationVerificationMessage",
    "consumerPid": "urn:uuid:b796fe3f-2055-40f6-a172-fd3d91164704",
    "providerPid": "urn:uuid:128098af-89fb-43f8-932e-4fe27cae69b9"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:uuid:b796fe3f-2055-40f6-a172-fd3d91164704",
    "providerPid": "urn:uuid:128098af-89fb-43f8-932e-4fe27cae69b9",
    "state": "VERIFIED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:65cec5b0-0cf3-4e89-9b64-7f90b2fc29b1",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Private",
    "state": "VERIFIED",
    "stateAttribute": null,
    "associatedAgentP

## 9. Finalization (Provider -> Consumer)

The Provider closes the negotiation. `FINALIZED` is terminal: the agreement can now be used to
request transfers.

In [12]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationFinalizationMessage",
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
}
response = requests.post(f"{data_space_provider}/dsp/current/negotiations/rpc/setup-finalization", headers=json_header, json=payload)
provider_process = negotiation_step(response, "FINALIZED")

HTTP 200 POST http://127.0.0.1:1200/dsp/current/negotiations/rpc/setup-finalization
{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcNegotiationResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcNegotiationFinalizationMessage",
    "consumerPid": "urn:uuid:b796fe3f-2055-40f6-a172-fd3d91164704",
    "providerPid": "urn:uuid:128098af-89fb-43f8-932e-4fe27cae69b9"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:uuid:b796fe3f-2055-40f6-a172-fd3d91164704",
    "providerPid": "urn:uuid:128098af-89fb-43f8-932e-4fe27cae69b9",
    "state": "FINALIZED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:72a5343b-0126-4918-ac86-82d3150abe9a",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Public",
    "state": "FINALIZED",
    "stateAttribute": null,
    "associatedAgent

## 10. Agreement and final state on both sides

The agreement is the last offer both sides accepted, now with `assigner` (Provider) and `assignee`
(Consumer). Both agents keep the same agreement id; it is the `agreementId` a transfer request
names. The management API of each agent shows its own view of the process.

In [13]:
agreement = provider_process["agreement"]
agreement_id = agreement["id"]
print(json.dumps(agreement, indent=2))

for name, agent_url, process in [("Consumer", data_space_consumer, consumer_process), ("Provider", data_space_provider, provider_process)]:
    view = requests.get(f"{agent_url}/api/v1/negotiation-agent/negotiation-processes/{process['id']}", headers=json_header).json()
    print(f"{name}: {view['state']}, agreement {view['agreement']['id']}")

{
  "id": "urn:agreement:9a31d5ab-908e-4617-90e6-ec71fa546a77",
  "userId": "system",
  "userRole": "/admin",
  "visibility": "Public",
  "negotiationAgentProcessId": "urn:negotiation-process:72a5343b-0126-4918-ac86-82d3150abe9a",
  "negotiationAgentMessageId": "urn:negotiation-message:ead56101-74f8-4bab-b1af-74166625944c",
  "consumerParticipantId": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ5cS12NXdJNFM2MTl2SnJITmpENDVra0owUGw4VGlBTzZYTnFJc1o4SDBaTEs5T2JoWTBTeGtEQ2JlZWd2RlZEU0MzM182N3lVZVpUWFI2SjFXUExHdHNtLU51SXJtR1RVRVgteFA0WnFLVGRFUTFSdUV0MExtMUhtdkdJWG1zLVJrTXRNVjcxRWt0MjNiZFYzX01WTjRUOUd5WnlaQTB6RDRlTFUxRTRTRzlRMGl4MGFRbC16eGdsVFRYQ2l4VTdCTzRXcHRkaUp3a0Jid2hMbnVqeXFzSmNvQXpPU000aXZtZlRTVW1Yd0Nrc1gySTFxVTNnVWdtUGFtbmEta3ZUb3FYZUk4ODNlRTFkSWNuUG53dmJlRmpfa0xuQ2JrNzhjNkVkRXZ0OVBMa3ByTWZfdmVxeVlOWjB0WlpYUGM5c0lRVlMyUG5xZkN5RUhsWWVuMXNXclEifQ",
  "providerParticipantId": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEa

In [14]:
# Marker read by the guards of the transfer notebooks
agreement_source = ("rpc", target_id)
print(f"agreement_id = {agreement_id}")

agreement_id = urn:agreement:9a31d5ab-908e-4617-90e6-ec71fa546a77


## Optional: termination

Either side may terminate a negotiation that is not finished yet with `setup-termination`
(`RpcNegotiationTerminationMessage`, with an optional `code` and `reason`), which moves it to
`TERMINATED` (DSP 8.2.6, 8.3.7). This one is `FINALIZED`, so the agent refuses: the cell shows the
`ContractNegotiationError` it answers.

In [15]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationTerminationMessage",
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "code": "TUTORIAL",
    "reason": ["The negotiation is already finalized"],
}
answer = show(requests.post(f"{data_space_consumer}/dsp/current/negotiations/rpc/setup-termination", headers=json_header, json=payload), ok=False)

HTTP 400 POST http://127.0.0.1:1100/dsp/current/negotiations/rpc/setup-termination
{
  "@context": [
    "https://w3id.org/dspace/2025/1/context.jsonld"
  ],
  "@type": "ContractNegotiationError",
  "consumerPid": "urn:uuid:b796fe3f-2055-40f6-a172-fd3d91164704",
  "providerPid": "urn:uuid:128098af-89fb-43f8-932e-4fe27cae69b9",
  "code": "400",
  "reason": [
    "state: ContractNegotiationTerminationMessage is not allowed from state FINALIZED"
  ]
}
